In [0]:
%run /Workspace/Users/ceresrain@yahoo.com/Butterfly_effect/short-form-butterfly/config/00_config/config

Config loaded.
Bronze: bootcamp_students.tiffani_ceresrain_bronze
Silver: bootcamp_students.tiffani_ceresrain_silver
Gold:   bootcamp_students.tiffani_ceresrain_gold
Films loaded: 50


In [0]:
# — Ingest IMDb ratings for film list
import pandas as pd
import requests
from io import BytesIO
import gzip

print("Downloading IMDb basics...")
basics_url = "https://datasets.imdbws.com/title.basics.tsv.gz"
ratings_url = "https://datasets.imdbws.com/title.ratings.tsv.gz"

basics = pd.read_csv(basics_url, sep='\t', na_values='\\N', low_memory=False)
ratings = pd.read_csv(ratings_url, sep='\t', na_values='\\N')

print(f"Basics loaded: {len(basics):,} rows")
print(f"Ratings loaded: {len(ratings):,} rows")

# Filter to movies only, merge
movies = (basics[basics['titleType'] == 'movie']
    .merge(ratings, on='tconst')
)

# Extract just our film list
film_titles = [f['title'] for f in FILM_LIST]

df_imdb = movies[movies['primaryTitle'].isin(film_titles)][
    ['tconst', 'primaryTitle', 'startYear', 'genres', 
     'runtimeMinutes', 'averageRating', 'numVotes']
].copy()

df_imdb = df_imdb.rename(columns={
    'primaryTitle': 'film',
    'startYear':    'release_year',
    'averageRating':'imdb_rating',
    'numVotes':     'vote_count',
    'runtimeMinutes': 'runtime_minutes'
})

# Keep only rows with enough votes to be meaningful
df_imdb = df_imdb[df_imdb['vote_count'] > 10000]

print(f"\nFilms matched: {len(df_imdb)}")
display(df_imdb)

Basics loaded: 12,365,381 rows
Ratings loaded: 1,649,242 rows

Films matched: 53


tconst,film,release_year,genres,runtime_minutes,imdb_rating,vote_count
tt0061452,Casino Royale,1967.0,Comedy,131,5.0,34754
tt0087182,Dune,1984.0,"Action,Adventure,Sci-Fi",137,6.2,192037
tt0372784,Batman Begins,2005.0,"Action,Crime,Drama",140,8.2,1703636
tt0381061,Casino Royale,2006.0,"Action,Adventure,Thriller",144,8.0,740706
tt0382932,Ratatouille,2007.0,"Adventure,Animation,Comedy",111,8.1,935005
tt0388795,Brokeback Mountain,2005.0,"Drama,Romance",134,7.7,414988
tt0407887,The Departed,2006.0,"Crime,Drama,Thriller",151,8.5,1545039
tt0439572,The Flash,2023.0,"Action,Adventure,Fantasy",144,6.6,245890
tt0468569,The Dark Knight,2008.0,"Action,Crime,Drama",152,9.1,3145602
tt0469494,There Will Be Blood,2007.0,Drama,158,8.2,703599


In [0]:
# Find duplicate matches
duplicates = df_imdb[df_imdb.duplicated(subset=['film'], keep=False)]
print(duplicates[['film', 'tconst', 'release_year', 'vote_count']].to_string())

                 film     tconst  release_year  vote_count
32424   Casino Royale  tt0061452        1967.0       34754
128356  Casino Royale  tt0381061        2006.0      740706
179311   The Revenant  tt1336006        2009.0       11735
204854   The Revenant  tt1663202        2015.0      943925
215958          Joker  tt1918886        2012.0       10331
322290          Joker  tt7286456        2019.0     1680606


In [0]:
# — Remove duplicate matches, keep correct versions
df_imdb = df_imdb[~(
    (df_imdb['film'] == 'The Revenant') & (df_imdb['tconst'] == 'tt1336006')
)]
df_imdb = df_imdb[~(
    (df_imdb['film'] == 'Joker') & (df_imdb['tconst'] == 'tt1918886')
)]

print(f"Films after dedup: {len(df_imdb)}")
display(df_imdb[['film', 'tconst', 'release_year', 'vote_count']])

Films after dedup: 51


film,tconst,release_year,vote_count
Casino Royale,tt0061452,1967.0,34754
Dune,tt0087182,1984.0,192037
Batman Begins,tt0372784,2005.0,1703636
Casino Royale,tt0381061,2006.0,740706
Ratatouille,tt0382932,2007.0,935005
Brokeback Mountain,tt0388795,2005.0,414988
The Departed,tt0407887,2006.0,1545039
The Flash,tt0439572,2023.0,245890
The Dark Knight,tt0468569,2008.0,3145602
There Will Be Blood,tt0469494,2007.0,703599


In [0]:
# Fix remaining duplicates
df_imdb = df_imdb[~(
    (df_imdb['film'] == 'Casino Royale') & (df_imdb['tconst'] == 'tt0061452')
)]
df_imdb = df_imdb[~(
    (df_imdb['film'] == 'Dune') & (df_imdb['tconst'] == 'tt0087182')
)]

print(f"Films after dedup: {len(df_imdb)}")

Films after dedup: 49


In [0]:
# Find which film is missing
matched_films = df_imdb['film'].tolist()
missing = [f['title'] for f in FILM_LIST if f['title'] not in matched_films]
print("Missing films:", missing)

Missing films: ['Five Nights at Freddys']


In [0]:
film_titles = [f['title'] for f in FILM_LIST]

In [0]:
# Keep correct versions only
df_imdb = df_imdb[~(
    (df_imdb['film'] == 'The Revenant') & (df_imdb['tconst'] == 'tt1336006')
)]
df_imdb = df_imdb[~(
    (df_imdb['film'] == 'Joker') & (df_imdb['tconst'] == 'tt1918886')
)]

print(f"Films after dedup: {len(df_imdb)}")

Films after dedup: 38


In [0]:
# — Save to Bronze Delta table
df_spark = spark.createDataFrame(df_imdb)

(df_spark
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(f"{BRONZE_PATH}.imdb_ratings_raw")
)

print(f"✅ Saved to {BRONZE_PATH}.imdb_ratings_raw")
print(f"   Rows written: {df_spark.count()}")

display(spark.table(f"{BRONZE_PATH}.imdb_ratings_raw").limit(5))

✅ Saved to bootcamp_students.tiffani_ceresrain_bronze.imdb_ratings_raw
   Rows written: 49


tconst,film,release_year,genres,runtime_minutes,imdb_rating,vote_count
tt0372784,Batman Begins,2005.0,"Action,Crime,Drama",140,8.2,1703636
tt0381061,Casino Royale,2006.0,"Action,Adventure,Thriller",144,8.0,740706
tt0382932,Ratatouille,2007.0,"Adventure,Animation,Comedy",111,8.1,935005
tt0388795,Brokeback Mountain,2005.0,"Drama,Romance",134,7.7,414988
tt0407887,The Departed,2006.0,"Crime,Drama,Thriller",151,8.5,1545039
